# Data Analytics Using Python — All-in-One Notebook

Audience: Entry-level students. This notebook teaches core DAP topics end-to-end with practical, well-commented code.

**Topics Covered:** Python basics; Pandas & NumPy; Visualization; EDA; Data prep; Statistics; Hypothesis testing; Confidence intervals; Scaling & Encoding; Train/Test & CV; KNN; Linear & Logistic Regression; Model evaluation; Feature engineering; PCA; Clustering; Time series; Pipelines; Best practices; Exercises.

In [ ]:
# Setup: import core libraries used throughout the course
# Tip: If an import fails, install via pip in your environment (e.g., `pip install pandas seaborn scikit-learn`).
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
sns.set(style='whitegrid', context='notebook')
plt.style.use('fivethirtyeight')
print('Libraries ready.')

## 1) Python Basics (variables, types, lists, dicts, loops, functions)

In [ ]:
# Variables and data types
x = 42                # integer
pi = 3.14159          # float
name = 'Alice'        # string
is_student = True     # boolean

# Lists (ordered, mutable)
scores = [88, 92, 79, 95]
scores.append(85)     # add an element

# Dictionaries (key-value pairs)
info = {'name': name, 'age': 25, 'student': is_student}

# Loops and conditionals
for s in scores:
    if s >= 90:
        print(f'High score: {s}')

# Functions encapsulate reusable logic
def add(a, b):
    return a + b

print('Sum:', add(3, 4))

## 2) Pandas DataFrames and Core Operations

In [ ]:
# Create a small DataFrame for demonstration
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Dana'],
    'Age': [25, 32, 37, np.nan],
    'Score': [88.5, 92.0, 79.5, 85.0],
    'Department': ['CS', 'Math', 'CS', 'Physics']
})

# Inspect and summarize
print(df.head())
print('
Describe:')
print(df.describe())

# Selection and filtering
cs_students = df[df['Department'] == 'CS']
print('
CS students:
', cs_students)

# Handling missing data
print('
Missing values per column:
', df.isna().sum())
df_filled = df.fillna({'Age': df['Age'].median()})
print('
After fillna median Age:
', df_filled)

# Groupby aggregation
dept_mean = df.groupby('Department')['Score'].mean()
print('
Mean Score by Department:
', dept_mean)

## 3) NumPy Arrays and Basic Statistics

In [ ]:
# Create and analyze a NumPy array
arr = np.array([1, 2, 3, 4, 5, 6])
print('Mean:', arr.mean(), 'Std:', arr.std(), 'Sum:', arr.sum())

# Broadcasting (vectorized operations)
print('Double each:', arr * 2)

## 4) Data Visualization (Matplotlib & Seaborn)

In [ ]:
# Basic Matplotlib line, scatter, and histogram
x = np.arange(0, 10)
y = x ** 2
plt.figure(figsize=(10, 3))
plt.plot(x, y, marker='o'); plt.title('Line Plot'); plt.xlabel('x'); plt.ylabel('y'); plt.show()

plt.figure(figsize=(10, 3))
plt.scatter(x, y, color='teal'); plt.title('Scatter Plot'); plt.show()

plt.figure(figsize=(10, 3))
samples = np.random.normal(loc=0, scale=1, size=200)
plt.hist(samples, bins=20, color='purple'); plt.title('Histogram'); plt.show()

# Seaborn pairplot using built-in iris dataset
iris = sns.load_dataset('iris')
sns.pairplot(iris, hue='species');

## 5) Exploratory Data Analysis (EDA) and Correlations

In [ ]:
# Summary statistics and correlation heatmap
print(iris.describe())
plt.figure(figsize=(8, 6))
sns.heatmap(iris.corr(numeric_only=True), annot=True, cmap='RdBu_r', center=0)
plt.title('Iris Correlation Matrix'); plt.show()

## 6) Statistical Foundations: Hypothesis Testing & Confidence Intervals

In [ ]:
# Two-sample t-test (Welch) on synthetic groups
np.random.seed(42)
group_A = np.random.normal(loc=5.0, scale=1.0, size=100)
group_B = np.random.normal(loc=5.5, scale=1.2, size=100)
t, p = stats.ttest_ind(group_A, group_B, equal_var=False)
print(f'T-statistic: {t:.3f}, p-value: {p:.4f}')

# 95% Confidence Interval for a mean (normal approximation)
sample_mean = group_A.mean()
sample_std = group_A.std(ddof=1)
n = len(group_A)
z = 1.96
ci_low = sample_mean - z * (sample_std / math.sqrt(n))
ci_high = sample_mean + z * (sample_std / math.sqrt(n))
print(f'95% CI for mean of group_A: ({ci_low:.3f}, {ci_high:.3f})')

## 7) Data Preparation: Scaling, Encoding, Outliers

In [ ]:
# Encoding categorical variables and scaling numerics
df_enc = pd.get_dummies(df_filled, columns=['Department'], drop_first=True)
num_cols = ['Age', 'Score']
scaler = StandardScaler()
df_enc[num_cols] = scaler.fit_transform(df_enc[num_cols])
print('Encoded & scaled:
', df_enc.head())

# Simple outlier detection via z-score
z_scores = np.abs(stats.zscore(df_filled[['Age', 'Score']].dropna()))
print('
Z-scores (Age, Score):
', z_scores[:5])

## 8) Train/Test Split and Cross-Validation

In [ ]:
# Use iris for classification (target = species setosa vs others)
X = iris.drop(columns=['species'])
y = (iris['species'] == 'setosa').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cross-validation with a simple pipeline (scaling + KNN)
pipe = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
print('CV accuracy (mean ± std):', cv_scores.mean().round(3), '±', cv_scores.std().round(3))

## 9) K-Nearest Neighbors (KNN) Classification

In [ ]:
# Fit and evaluate KNN
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, y_pred))
print('
Confusion matrix:
', confusion_matrix(y_test, y_pred))
print('
Report:
', classification_report(y_test, y_pred))

## 10) Linear Regression (Supervised Regression)

In [ ]:
# Synthetic linear data and LinearRegression
X_lin = np.array([[1], [2], [3], [4], [5]])
y_lin = np.array([2.1, 3.9, 6.0, 8.2, 9.9])
lin_model = LinearRegression().fit(X_lin, y_lin)
y_lin_pred = lin_model.predict(X_lin)
print('Slope:', lin_model.coef_, 'Intercept:', lin_model.intercept_)
print('R2:', r2_score(y_lin, y_lin_pred), 'RMSE:', np.sqrt(mean_squared_error(y_lin, y_lin_pred)).round(3))
plt.figure(figsize=(6,4)); plt.scatter(X_lin, y_lin, color='blue'); plt.plot(X_lin, y_lin_pred, color='red'); plt.title('Linear Regression Fit'); plt.show()

## 11) Logistic Regression (Binary Classification)

In [ ]:
# Predict setosa vs others with LogisticRegression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
probs = logreg.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, probs)
print('ROC-AUC:', auc.round(3))
fpr, tpr, _ = roc_curve(y_test, probs)
plt.figure(figsize=(6,4)); plt.plot(fpr, tpr); plt.plot([0,1],[0,1],'k--'); plt.title('ROC Curve'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.show()

## 12) Feature Engineering (dates, interactions, simple text)

In [ ]:
# Date parsing and components
dates = pd.Series(pd.to_datetime(['2024-01-01', '2024-01-15', '2024-02-01']))
date_df = pd.DataFrame({'date': dates})
date_df['year'] = date_df['date'].dt.year
date_df['month'] = date_df['date'].dt.month
date_df['day'] = date_df['date'].dt.day
print(date_df)

# Interaction feature example
df_fe = df_filled.copy()
df_fe['age_score_interaction'] = df_fe['Age'] * df_fe['Score']
print(df_fe[['Age','Score','age_score_interaction']])

# Simple text feature: string length
texts = pd.Series(['data science', 'python', 'analytics'])
text_len = texts.apply(len)
print('Text lengths:', list(text_len))

## 13) Dimensionality Reduction: PCA

In [ ]:
# Standardize features then apply PCA
X_std = StandardScaler().fit_transform(iris.drop(columns=['species']))
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)
print('Explained variance ratios:', pca.explained_variance_ratio_)
plt.figure(figsize=(6,4)); plt.scatter(X_pca[:,0], X_pca[:,1], c=iris['species'].map({'setosa':0,'versicolor':1,'virginica':2}), cmap='viridis'); plt.title('PCA (2 components)'); plt.show()

## 14) Unsupervised Learning: K-Means Clustering

In [ ]:
# Cluster iris features (first two for visualization)
X_vis = iris[['sepal_length','sepal_width']].values
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X_vis)
plt.figure(figsize=(6,4)); plt.scatter(X_vis[:,0], X_vis[:,1], c=labels, cmap='plasma'); plt.title('K-Means Clusters (k=3)'); plt.show()
print('Cluster centers:
', kmeans.cluster_centers_)

## 15) Time Series Basics (resampling, rolling)

In [ ]:
# Create a simple daily time series and compute rolling mean
dates = pd.date_range(start='2024-01-01', periods=60, freq='D')
values = pd.Series(np.random.randn(60).cumsum())
ts = pd.DataFrame({'date': dates, 'value': values}).set_index('date')
ts_resampled = ts.resample('W').mean()
ts['rolling_7'] = ts['value'].rolling(window=7).mean()
ts.plot(figsize=(10,4)); plt.title('Daily Series with 7-Day Rolling Mean'); plt.show()
ts_resampled.plot(figsize=(10,4)); plt.title('Weekly Resampled Mean'); plt.show()

## 16) Pipelines and Best Practices

In [ ]:
# Combine preprocessing and model into a single pipeline
pipe_lr = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(max_iter=1000))])
cv_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='roc_auc')
print('Pipeline ROC-AUC (mean ± std):', cv_lr.mean().round(3), '±', cv_lr.std().round(3))

# Reproducibility tips: set random_state, document assumptions, validate via CV
print('Best practices: set seeds (random_state), avoid leakage, use CV, explain assumptions.')

## 17) Practice Exercises
1. Load a new CSV and perform EDA (describe, plots, correlations).
2. Clean missing values, encode categories, scale numerics.
3. Train a KNN classifier; tune `k` using cross-validation.
4. Fit Linear Regression on a regression dataset; report R² and RMSE.
5. Apply PCA to reduce dimensions; plot explained variance.
6. Cluster with K-Means and interpret clusters.
7. Create a time series, resample weekly, compute a rolling mean.

### Exercise 1: Load a CSV and perform EDA (describe, plots, correlations)
Goal: practice reading data, inspecting structure, plotting distributions, and seeing relationships.
Steps: create a small dataset, write to CSV, read it back, EDA with describe, histograms, scatter, and correlation heatmap.

In [ ]:
# Create synthetic sales data and save to CSV
np.random.seed(1)
dates = pd.date_range('2024-01-01', periods=120, freq='D')
regions = np.random.choice(['North','South','East','West'], size=120)
products = np.random.choice(['A','B','C'], size=120)
units = np.random.poisson(lam=20, size=120)
price = np.random.uniform(10, 50, size=120).round(2)
sales = pd.DataFrame({'Date': dates, 'Region': regions, 'Product': products, 'Units': units, 'Price': price})
sales.to_csv('practice_sales.csv', index=False)

# Read CSV and EDA
sales_df = pd.read_csv('practice_sales.csv')
print(sales_df.head())
print('
Describe:')
print(sales_df.describe())

# Distribution plots
plt.figure(figsize=(10,4)); sns.histplot(sales_df['Units'], bins=20, color='steelblue'); plt.title('Units Distribution'); plt.show()

# Relationship: price vs units
plt.figure(figsize=(10,4)); sns.scatterplot(x='Price', y='Units', data=sales_df, alpha=0.7); plt.title('Price vs Units'); plt.show()

# Correlation heatmap (numerics only)
plt.figure(figsize=(6,5)); sns.heatmap(sales_df[['Units','Price']].corr(), annot=True, cmap='RdBu_r', center=0); plt.title('Correlation'); plt.show()

#### Exercise 1 Explanation
EDA inspects distributions and relationships. The histogram shows how `Units` vary; the scatter plot reveals the association between `Price` and `Units`; the correlation heatmap quantifies the linear relationship between the two numeric variables.


### Exercise 2: Clean missing values, encode categories, scale numerics
Goal: practice data cleaning and preprocessing for modeling.
Steps: introduce some NaNs, impute, one-hot encode, and scale numeric columns.

In [ ]:
# Introduce missing values artificially
sales_df.loc[sales_df.sample(frac=0.1, random_state=42).index, 'Units'] = np.nan

# Impute Units with median
median_units = sales_df['Units'].median()
sales_df['Units'] = sales_df['Units'].fillna(median_units)

# One-hot encode categorical variables
encoded = pd.get_dummies(sales_df, columns=['Region','Product'], drop_first=True)

# Scale numeric features
num_cols = ['Units','Price']
scaler = StandardScaler()
encoded[num_cols] = scaler.fit_transform(encoded[num_cols])
print(encoded.head())

#### Exercise 2 Explanation
Median imputation replaces missing `Units` with a robust central value. One-hot encoding converts categorical `Region` and `Product` into numeric indicators without imposing ordinality. Standardization centers and scales numeric features, improving performance for distance-based models.


### Exercise 3: Train a KNN classifier; tune k with cross-validation
Goal: build a classifier and see how `k` affects accuracy.
Steps: use iris, pipeline with scaling, and loop over k to compute CV accuracy.

In [ ]:
# Prepare iris features/target
X_cls = iris.drop(columns=['species'])
y_cls = iris['species']

# Encode species to integers for KNN
le = LabelEncoder()
y_cls_enc = le.fit_transform(y_cls)

# Tune k via cross-validation
results = []
for k in [3,5,7,9,11]:
    pipe_knn = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k))])
    scores = cross_val_score(pipe_knn, X_cls, y_cls_enc, cv=5, scoring='accuracy')
    results.append((k, scores.mean(), scores.std()))

print('k, mean accuracy, std:')
for r in results:
    print(r)

#### Exercise 3 Explanation
Cross-validation estimates generalization. Smaller `k` tends to fit local noise (overfit), larger `k` smooths decision boundaries (potential underfit). The reported mean accuracy across folds helps select a balanced `k`.


### Exercise 4: Fit Linear Regression; report R² and RMSE
Goal: practice supervised regression evaluation.
Steps: use a standard dataset (diabetes), fit, predict, compute metrics.

In [ ]:
# Linear regression on diabetes dataset
from sklearn.datasets import load_diabetes
data = load_diabetes()
X_reg = data.data
y_reg = data.target
Xtr, Xte, ytr, yte = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
lr = LinearRegression().fit(Xtr, ytr)
yp = lr.predict(Xte)
print('R2:', r2_score(yte, yp).round(3))
print('RMSE:', np.sqrt(mean_squared_error(yte, yp)).round(3))

#### Exercise 4 Explanation
R² indicates the proportion of target variance explained by the model; RMSE is the average magnitude of prediction error. Together they summarize fit quality and error scale on the test set.


### Exercise 5: Apply PCA and plot explained variance
Goal: reduce dimensions and understand variance captured.
Steps: standardize features, fit PCA, plot a scree chart of variance ratios.

In [ ]:
# PCA scree plot
if 'X_cls' not in globals():
    X_cls = iris.drop(columns=['species'])
X_std2 = StandardScaler().fit_transform(X_cls)
pca_full = PCA().fit(X_std2)
var_ratio = pca_full.explained_variance_ratio_
plt.figure(figsize=(8,4)); plt.plot(np.arange(1, len(var_ratio)+1), var_ratio, marker='o'); plt.title('PCA Explained Variance Ratio'); plt.xlabel('Component'); plt.ylabel('Variance Ratio'); plt.show()

#### Exercise 5 Explanation
The scree plot shows how much variance each principal component explains. A sharp drop suggests a natural cutoff; retaining the top components captures most structure while reducing dimensionality.


### Exercise 6: Cluster with K-Means and interpret clusters
Goal: perform unsupervised clustering and relate clusters to known labels.
Steps: cluster standardized iris features and compare cluster labels to species via crosstab.

In [ ]:
# K-Means on standardized iris features; interpret via crosstab
if 'y_cls' not in globals():
    y_cls = iris['species']
kmeans2 = KMeans(n_clusters=3, random_state=0)
labels2 = kmeans2.fit_predict(X_std2)
ct = pd.crosstab(pd.Series(labels2, name='Cluster'), pd.Series(y_cls, name='Species'))
print(ct)

#### Exercise 6 Explanation
The crosstab aligns discovered clusters with species labels. Strong diagonal counts indicate clusters correspond well to true classes; mixed counts imply overlap or non-spherical clusters.


### Exercise 7: Create a time series, resample weekly, compute a rolling mean
Goal: practice time series resampling and smoothing.
Steps: generate daily data, resample to weekly mean, compute and plot a rolling mean.

#### Exercise 7 Explanation
Rolling means smooth short-term noise, revealing underlying trends; weekly resampling aggregates daily observations to a coarser cadence. Comparing both views helps distinguish signal from noise.


In [ ]:
# Time series exercise
dates2 = pd.date_range('2024-03-01', periods=120, freq='D')
vals2 = pd.Series(np.random.randn(120).cumsum())
ts2 = pd.DataFrame({'date': dates2, 'value': vals2}).set_index('date')
ts2['rolling_14'] = ts2['value'].rolling(14).mean()
ts2_resamp = ts2.resample('W').mean()
ts2.plot(figsize=(10,4)); plt.title('Daily Series with 14-Day Rolling Mean'); plt.show()
ts2_resamp.plot(figsize=(10,4)); plt.title('Weekly Resampled Mean'); plt.show()

## End-Semester Exam-Style Questions (10) with Solutions
The following questions simulate exam tasks. Each includes a solution and an explanation.


### Q1: Stratified CV with Logistic Regression on Iris
Compute 5-fold stratified cross-validated macro F1 using a pipeline (scaling + logistic regression).


In [ ]:
from sklearn.model_selection import StratifiedKFold
Xq1 = iris.drop(columns=['species'])
yq1 = iris['species']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_q1 = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(max_iter=1000))])
cv_q1 = cross_val_score(pipe_q1, Xq1, yq1, cv=skf, scoring='f1_macro')
print('F1-macro (mean ± std):', cv_q1.mean().round(3), '±', cv_q1.std().round(3))


Stratification preserves class proportions across folds, yielding stable estimates. Macro F1 averages F1 across classes, emphasizing balanced performance rather than favoring the majority class.


### Q2: Chi-Square Test of Independence (Region vs Product)
Test whether `Region` and `Product` are independent using a contingency table and chi-square statistic.


In [ ]:
sales_exam = pd.read_csv('practice_sales.csv')
ct = pd.crosstab(sales_exam['Region'], sales_exam['Product'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print('Chi2:', round(chi2,3), 'p:', round(p,4), 'dof:', dof)


A small p-value suggests `Region` and `Product` are associated rather than independent. The test compares observed counts to those expected under independence.


### Q3: 95% CI for Difference of Means
Estimate a 95% confidence interval for the difference of two normal means.


In [ ]:
np.random.seed(0)
g1 = np.random.normal(50, 10, size=100)
g2 = np.random.normal(53, 10, size=100)
diff = g2.mean() - g1.mean()
se = math.sqrt(g1.var(ddof=1)/len(g1) + g2.var(ddof=1)/len(g2))
z = 1.96
print('Diff mean:', round(diff,3), '95% CI:', (round(diff - z*se,3), round(diff + z*se,3)))


The interval quantifies uncertainty around the mean difference. If it excludes 0, the groups differ significantly at the 5% level under normal assumptions.


### Q4: Scaling Choice Impact on KNN Accuracy
Compare StandardScaler vs MinMaxScaler for KNN classification on iris.


In [ ]:
X_s = iris.drop(columns=['species'])
y_s = LabelEncoder().fit_transform(iris['species'])
results_scaling = {}
for scaler in [MinMaxScaler(), StandardScaler()]:
    pipe_scaling = Pipeline([('scaler', scaler), ('knn', KNeighborsClassifier(n_neighbors=5))])
    scores_scaling = cross_val_score(pipe_scaling, X_s, y_s, cv=5, scoring='accuracy')
    results_scaling[type(scaler).__name__] = (scores_scaling.mean().round(3), scores_scaling.std().round(3))
print(results_scaling)


Distance-based models depend on feature scales. Different scalers can change neighbor relationships; the comparison shows which normalization yields better accuracy for this dataset.


### Q5: K-Means Elbow Method
Use inertia across k=1..10 on standardized iris features to choose k.


In [ ]:
X_elbow = StandardScaler().fit_transform(iris.drop(columns=['species']))
ks = range(1, 11)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=0)
    km.fit(X_elbow)
    inertias.append(km.inertia_)
plt.figure(figsize=(6,4)); plt.plot(list(ks), inertias, marker='o'); plt.title('Elbow Method'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.show()


An elbow (sharp bend) indicates diminishing returns from adding clusters. Choosing k near the elbow balances compactness with parsimony.


### Q6: Feature Selection with SelectKBest (ANOVA F)
Evaluate accuracy for k=2,3,4 selected features using a pipeline.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
X_sel = iris.drop(columns=['species'])
y_sel = LabelEncoder().fit_transform(iris['species'])
for k in [2,3,4]:
    pipe_sel = Pipeline([('scaler', StandardScaler()), ('select', SelectKBest(score_func=f_classif, k=k)), ('logreg', LogisticRegression(max_iter=1000))])
    scores_sel = cross_val_score(pipe_sel, X_sel, y_sel, cv=5, scoring='accuracy')
    print(k, scores_sel.mean().round(3), scores_sel.std().round(3))


Filter-based selection ranks features by discriminative power. The best k balances information retention and noise reduction, improving classifier performance.


### Q7: PCA + Logistic Regression Tuning
Assess accuracy across PCA components 1–4 on iris using a pipeline.


In [ ]:
X_p = iris.drop(columns=['species'])
y_p = LabelEncoder().fit_transform(iris['species'])
comps = [1,2,3,4]
means = []
for n in comps:
    pipe_p = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=n)), ('logreg', LogisticRegression(max_iter=1000))])
    scores_p = cross_val_score(pipe_p, X_p, y_p, cv=5, scoring='accuracy')
    means.append(scores_p.mean())
plt.figure(figsize=(6,4)); plt.plot(comps, means, marker='o'); plt.title('Accuracy vs PCA Components'); plt.xlabel('n_components'); plt.ylabel('Accuracy'); plt.show()


PCA can reduce noise and multicollinearity. The curve shows how dimensionality affects classification; too few components underfit, too many reintroduce noise.


### Q8: Autocorrelation of a Daily Time Series
Compute and plot autocorrelation for lags 1–20 to assess temporal dependence.


In [ ]:
ts_exam = ts2 if 'ts2' in globals() else pd.DataFrame({'date': pd.date_range('2024-03-01', periods=120, freq='D'), 'value': pd.Series(np.random.randn(120).cumsum())}).set_index('date')
lags = range(1, 21)
acfs = [ts_exam['value'].autocorr(lag=l) for l in lags]
plt.figure(figsize=(6,4)); plt.bar(list(lags), acfs); plt.title('Autocorrelation'); plt.xlabel('Lag'); plt.ylabel('ACF'); plt.show()


High autocorrelation at small lags indicates persistence; near-zero values suggest weak temporal dependence. Patterns inform smoothing and forecasting choices.


### Q9: ROC Curve for KNN (Binary Setosa vs Others)
Train KNN with scaling and plot ROC; report AUC.


In [ ]:
X_b = iris.drop(columns=['species'])
y_b = (iris['species'] == 'setosa').astype(int)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.2, random_state=42)
pipe_b = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
pipe_b.fit(X_train_b, y_train_b)
probs_b = pipe_b.predict_proba(X_test_b)[:,1]
fpr_b, tpr_b, _ = roc_curve(y_test_b, probs_b)
auc_b = roc_auc_score(y_test_b, probs_b)
print('ROC-AUC:', auc_b.round(3))
plt.figure(figsize=(6,4)); plt.plot(fpr_b, tpr_b); plt.plot([0,1],[0,1],'k--'); plt.title('KNN ROC'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.show()


AUC summarizes ranking quality across thresholds. Curves above the diagonal indicate discriminatory power; higher AUC reflects better separability of setosa vs others.


### Q10: CV R² for Scaled Linear Regression (Diabetes)
Evaluate 5-fold cross-validated R² using a pipeline.


In [ ]:
from sklearn.datasets import load_diabetes
X_d = load_diabetes().data
y_d = load_diabetes().target
pipe_d = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
scores_d = cross_val_score(pipe_d, X_d, y_d, cv=5, scoring='r2')
print('CV R2 (mean ± std):', scores_d.mean().round(3), '±', scores_d.std().round(3))


Cross-validated R² estimates generalization for regression. Scaling helps stabilize coefficient estimation; the mean and spread across folds indicate model reliability.
